# UAV3DCrop 向け Depth Anything V2 Small Metric Depth

CPU準備・監査とColab GPU処理を分離したNotebookです。上から順に実行し、「すべて実行」は使わないでください。データ、モデル重み、GPU利用はそれぞれ明示ゲートを開いたときだけ行います。独立testはモデル選択が終わるまで読みません。

最初にREADME_ja.mdを読み、このフォルダー全体をGoogle Driveの MyDrive/DepthAnythingV2-UAV/uav_metric_finetuning に置いてください。config/verified_planも必要です。

**0. Drive接続・状態復元**は最初のコードセルです。CPU→GPU切替後はこのセルを再実行してください。データ取得や画素監査は自動で実行しません。

## 0. Drive接続・状態復元

GPU切替後にも、次のコードセルから開始してください。

In [ ]:
SESSION_READY = False
from pathlib import Path
import json, shutil, subprocess, sys
from datetime import datetime, timezone
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/DepthAnythingV2-UAV')
CODE_DIR = PROJECT_DIR / 'uav_metric_finetuning'
PLAN_DIR = CODE_DIR / 'config' / 'verified_plan'
GENERATED_PLAN_DIR = PROJECT_DIR / 'work' / 'uav_metric_finetuning' / 'remote_plan_generated'
DRIVE_DATA_DIR = PROJECT_DIR / 'data'
DRIVE_RGB_ROOT = DRIVE_DATA_DIR / 'uav3dcrop_rgb'
DRIVE_DEPTH_ROOT = DRIVE_DATA_DIR / 'uav3dcrop_depth'
LOCAL_DATA_DIR = Path('/content/uav3dcrop')
LOCAL_RGB_ROOT, LOCAL_DEPTH_ROOT = LOCAL_DATA_DIR / 'rgb', LOCAL_DATA_DIR / 'depth'
RUNS_DIR = PROJECT_DIR / 'runs'
MODEL_CACHE_DIR = PROJECT_DIR / 'model_cache'
PREFLIGHT_DIR = PROJECT_DIR / 'preflight'
WORK_DIR = PROJECT_DIR / 'work' / 'uav_metric_finetuning'
for directory in (PROJECT_DIR, DRIVE_DATA_DIR, RUNS_DIR, MODEL_CACHE_DIR, PREFLIGHT_DIR, WORK_DIR):
    directory.mkdir(parents=True, exist_ok=True)
if not (CODE_DIR / 'train.py').is_file():
    raise FileNotFoundError(f'コードをDriveへ配置してください: {CODE_DIR}')
if not (PLAN_DIR / 'plan_summary.json').is_file():
    raise FileNotFoundError(f'計画ファイルがありません: {PLAN_DIR}')
sys.path.insert(0, str(CODE_DIR))
ACTIVE_RGB_ROOT, ACTIVE_DEPTH_ROOT = DRIVE_RGB_ROOT, DRIVE_DEPTH_ROOT
print('Project:', PROJECT_DIR, '| Code:', CODE_DIR, '| Plan:', PLAN_DIR)
try:
    print('Drive reported free GiB:', round(shutil.disk_usage(PROJECT_DIR).free / 1024**3, 2))
except OSError as exc:
    print('Drive free space unavailable:', exc)
print('Colab /content free GiB:', round(shutil.disk_usage('/content').free / 1024**3, 2))

# ランタイム変更・セッション終了後も、この初期化セルから復元できます。
import importlib, runtime_state as persistent_state
importlib.reload(persistent_state)
from runtime_state import (load_settings, save_settings, validate_settings, find_report,
                           preparation_context, dataset_status)
SETTINGS_PATH = WORK_DIR / 'session_settings.json'
TRAIN_CONFIG = load_settings(SETTINGS_PATH)
globals().update(TRAIN_CONFIG)
GT_SEED, GT_MAX_PIXELS = 42, 20000
MODEL_PATHS, PREFLIGHT_REPORTS = {}, {}
RUN_DOWNLOAD = ALLOW_MODEL_DOWNLOAD = RUN_GPU_PREFLIGHT = RUN_TRAINING = False
RUN_INDEPENDENT_TEST = RUN_STAGE_TO_LOCAL = RUN_DATA_AUDIT = RUN_GT_ANALYSIS = False

def audit_context_args():
    return dict(manifests={s: PLAN_DIR / f'{s}.csv' for s in ('train','validation','independent_test')},
                rgb_root=DRIVE_RGB_ROOT, depth_root=DRIVE_DEPTH_ROOT,
                physical_splits=['train','validation'], kind='audit',
                extra_files={'plan_summary': PLAN_DIR/'plan_summary.json',
                             'existing_eval': PROJECT_DIR/'results/uav3dcrop/all_selected_images.csv'})

def gt_context_args():
    return dict(manifests={'train': PLAN_DIR/'train.csv'}, rgb_root=None,
                depth_root=DRIVE_DEPTH_ROOT, physical_splits=['train'], kind='gt',
                parameters={'seed':GT_SEED,'max_pixels_per_frame':GT_MAX_PIXELS})

def refresh_preparation_state():
    global AUDIT_OK, AUDIT_REPORT, AUDIT_PATH, GT_REPORT, GT_REPORT_PATH
    AUDIT_PATH, AUDIT_REPORT, audit_reason = find_report(WORK_DIR, 'data_audit_*.json', **audit_context_args())
    AUDIT_OK = AUDIT_REPORT is not None
    GT_REPORT_PATH, GT_REPORT, gt_reason = find_report(WORK_DIR, 'gt_distribution_train_*.json', **gt_context_args())
    print('監査:', '再利用可能' if AUDIT_OK else '未完了・再監査が必要', audit_reason)
    print('GT解析:', '再利用可能' if GT_REPORT else '未完了・再解析が必要', gt_reason)
    return AUDIT_OK, GT_REPORT is not None

plan_summary = json.loads((PLAN_DIR/'plan_summary.json').read_text(encoding='utf-8'))
refresh_preparation_state()
SESSION_READY = True
print('初期化完了。Driveの設定を復元:', SETTINGS_PATH if SETTINGS_PATH.exists() else '既定の試験設定')
print('GPU切替後は環境・設定確認・モデルcacheセルを再実行し、GPU事前計測へ進んでください。')


## 1. CPU: 環境確認

CUDAがなくても停止しません。このセルはGPUを使わず、ColabのPyTorchも更新しません。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

import importlib.util
INSTALL_EXTRA_DEPS = False  # 不足時のみTrue。requirementsにTorchは含めていません。
if INSTALL_EXTRA_DEPS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(CODE_DIR / 'requirements-colab.txt')], check=True)
missing = [m for m in ('torch','transformers','huggingface_hub','cv2','tifffile','safetensors') if importlib.util.find_spec(m) is None]
if missing:
    print('不足モジュール:', missing, '必要ならINSTALL_EXTRA_DEPS=Trueでこのセルを再実行')
else:
    import torch, transformers, huggingface_hub
    print('Python', sys.version.split()[0], '| Torch', torch.__version__, '| Torch CUDA build', torch.version.cuda)
    print('Transformers', transformers.__version__, '| HF Hub', huggingface_hub.__version__)
    print('CUDA available (information only):', torch.cuda.is_available())

## 2. CPU: 分割計画と転送量

通常は同梱したrevision固定の計画を再利用します。再照会は小さなtransforms.jsonとファイルメタデータのみで、RGB・深度本体は取得しません。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_METADATA_PLAN = False
if RUN_METADATA_PLAN:
    if GENERATED_PLAN_DIR.exists() and any(GENERATED_PLAN_DIR.iterdir()):
        raise FileExistsError(f'上書き防止。別フォルダーを使ってください: {GENERATED_PLAN_DIR}')
    GENERATED_PLAN_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, str(CODE_DIR/'build_remote_plan.py'), '--output-dir', str(GENERATED_PLAN_DIR)], check=True)
    PLAN_DIR = GENERATED_PLAN_DIR
plan_summary = json.loads((PLAN_DIR/'plan_summary.json').read_text(encoding='utf-8'))
print('RGB/depth dataset commits:', plan_summary['rgb_revision'], plan_summary['depth_revision'])
for split, count in plan_summary['selected_pair_counts'].items():
    info = plan_summary['selected_bytes_by_split'][split]
    print(f'{split}: {count} pairs, {info["bytes"]:,} bytes ({info["gib"]:.3f} GiB)')
for split, dist in plan_summary['split_distributions'].items():
    print(split, '| crops:', dist['crops'], '| views:', dist['views'], '| days:', dist['days'])
print('Metadata missing candidates:')
for scene in plan_summary.get('scene_metadata', []):
    for item in scene.get('missing_candidate_details', []):
        print(scene['split'], scene['scene'], item)
train_val_bytes = sum(int(plan_summary['selected_bytes_by_split'][s]['bytes']) for s in ('train','validation'))
train_val_gib = train_val_bytes / 1024**3
print(f'train+validation: {train_val_bytes:,} bytes = {train_val_gib:.2f} GiB')
print(f'Drive + local copy needs about {2*train_val_gib:.2f} GiB raw; target at least {2*train_val_gib*1.1:.2f} GiB free.')

## 3. CPU: データ取得ゲート（初期値OFF）

取得先はDriveです。必要量・空き容量を確認後、実行する場合はRUN_DOWNLOAD=Trueに加え、確認欄へ計画の正確なbyte数を入力します。同サイズの既存ファイルは再利用します。independent_testは最終モデル確定後まで取得しません。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_DOWNLOAD = False
SPLITS_TO_DOWNLOAD = ['train', 'validation']  # independent_testは最終評価まで無効
CONFIRM_PLAN_TOTAL_BYTES = None  # train+validation: 7850330776
status = dataset_status(PLAN_DIR, DRIVE_RGB_ROOT, DRIVE_DEPTH_ROOT, SPLITS_TO_DOWNLOAD)
print('取得状況:', status)
if status['conflicting_files']:
    raise RuntimeError('サイズが異なる既存ファイルがあります。上書きせず、保存先を確認してください。')
if RUN_DOWNLOAD:
    if CONFIRM_PLAN_TOTAL_BYTES != status['planned_bytes']:
        raise RuntimeError('転送しません。計画とDriveの空きを確認後、CONFIRM_PLAN_TOTAL_BYTESを入力してください。')
    subprocess.run([sys.executable,str(CODE_DIR/'fetch_selected_data.py'),'--plan-dir',str(PLAN_DIR),
        '--rgb-root',str(DRIVE_RGB_ROOT),'--depth-root',str(DRIVE_DEPTH_ROOT),
        '--splits',*SPLITS_TO_DOWNLOAD,'--download','--confirm-total-bytes',str(status['planned_bytes'])],check=True)
    status = dataset_status(PLAN_DIR, DRIVE_RGB_ROOT, DRIVE_DEPTH_ROOT, SPLITS_TO_DOWNLOAD)
if status['complete']:
    print(f"取得完了: {status['complete_pairs']}組、{status['present_files']}ファイルの存在とサイズを確認しました。再ダウンロードは不要です。")
else:
    print(f"未取得・不完全: {status['missing_files']}ファイル。実行する場合だけRUN_DOWNLOAD=Trueにしてください。")


## 4. CPU: DriveからローカルSSDへ任意コピー

Drive直読み込みが遅いときだけ/contentへコピーします。必要容量を計算し、空きが足りなければ停止します。Driveの元ファイルは保持されます。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_STAGE_TO_LOCAL = False
STAGE_SPLITS = ['train', 'validation']
if RUN_STAGE_TO_LOCAL:
    subprocess.run([sys.executable,str(CODE_DIR/'stage_selected_data.py'),'--plan-dir',str(PLAN_DIR),
        '--source-rgb-root',str(DRIVE_RGB_ROOT),'--source-depth-root',str(DRIVE_DEPTH_ROOT),
        '--target-rgb-root',str(LOCAL_RGB_ROOT),'--target-depth-root',str(LOCAL_DEPTH_ROOT),
        '--splits',*STAGE_SPLITS,'--stage'],check=True)
    ACTIVE_RGB_ROOT, ACTIVE_DEPTH_ROOT = LOCAL_RGB_ROOT, LOCAL_DEPTH_ROOT
else: ACTIVE_RGB_ROOT, ACTIVE_DEPTH_ROOT = DRIVE_RGB_ROOT, DRIVE_DEPTH_ROOT
print('Active dataset roots:', ACTIVE_RGB_ROOT, ACTIVE_DEPTH_ROOT)

## 5. CPU: 対応・画素・漏洩監査

3つのmanifestすべてでscene/day分割漏洩を確認します。物理ファイルとTIFF画素は既定でtrain/validationだけを監査します。未取得ファイルがあれば不足数を報告し、画素監査をスキップします。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_DATA_AUDIT = False
FORCE_REAUDIT = False
refresh_preparation_state()
if AUDIT_OK and not FORCE_REAUDIT:
    print('正常な監査結果を再利用します:', AUDIT_PATH)
elif RUN_DATA_AUDIT:
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    audit_path = WORK_DIR/f'data_audit_{stamp}.json'
    subprocess.run([sys.executable,str(CODE_DIR/'audit_data.py'),
        '--train',str(PLAN_DIR/'train.csv'),'--validation',str(PLAN_DIR/'validation.csv'),
        '--independent-test',str(PLAN_DIR/'independent_test.csv'),
        '--physical-splits','train','validation','--plan-summary',str(PLAN_DIR/'plan_summary.json'),
        '--rgb-root',str(DRIVE_RGB_ROOT),'--depth-root',str(DRIVE_DEPTH_ROOT),'--output',str(audit_path)],check=True)
    refresh_preparation_state()
    print('監査結果:', audit_path, '| pass:', AUDIT_OK)
else:
    print('データ取得後、RUN_DATA_AUDIT=Trueで監査してください。未取得なら不足数を表示してスキップします。')


## 6. CPU: train-only GT深度分布

GTを読み込むため、データ取得と監査の後に実行します。解析対象はtrain manifestだけです。max_depthの候補を出しますが、分布を確認してから設定してください。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_GT_ANALYSIS = False
FORCE_GT_ANALYSIS = False
refresh_preparation_state()
if GT_REPORT and not FORCE_GT_ANALYSIS:
    print('正常なtrain-only GT解析を再利用:', GT_REPORT_PATH)
elif RUN_GT_ANALYSIS:
    if not AUDIT_OK:
        raise RuntimeError('データ取得と監査passが必要です。先に監査セルを実行してください。')
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output = WORK_DIR/f'gt_distribution_train_{stamp}.json'
    subprocess.run([sys.executable,str(CODE_DIR/'analyze_gt.py'),'--manifest',str(PLAN_DIR/'train.csv'),
        '--depth-root',str(DRIVE_DEPTH_ROOT),'--seed',str(GT_SEED),
        '--max-pixels-per-frame',str(GT_MAX_PIXELS),'--output',str(output)],check=True)
    refresh_preparation_state()
if GT_REPORT:
    print('train GT分位点 (m):', GT_REPORT['quantiles_m'])
    print('max_depth候補 (m):', GT_REPORT['suggested_max_depth_m'])
    print('範囲・単位を確認後、設定セルのOVERRIDESでMAX_DEPTH_Mを指定し保存してください。')
else:
    print('RUN_GT_ANALYSIS=Trueでtrainだけを解析してください。max_depthはまだ設定しません。')


## 7. CPU: 学習条件

既定は1方式・1epoch・518解像度です。比較方式を試すときは1方式ずつ切り替えます。両方の連続実行はTRAINING_MODESへ2方式を明記した場合だけです。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

# 初期値: relative_encoder / trial / EPOCHS=1 / 518 / batch1 / accum4 / AMP / seed42
# 初回GT確認後の例: OVERRIDES={'MAX_DEPTH_M': 確認した値}
# 本学習の例: {'RUN_KIND':'full','EPOCHS':20,'RUN_LABEL':'full_s42_v1','RESUME_TRAINING':False}
# 同じ本学習の中断再開: {'RESUME_TRAINING':True}（他の設定はそのまま）
OVERRIDES = {}
SAVE_SETTINGS = False  # 編集をDriveに保存するときだけTrue
TRAIN_CONFIG = validate_settings(TRAIN_CONFIG | OVERRIDES)
globals().update(TRAIN_CONFIG)
if SAVE_SETTINGS:
    save_settings(SETTINGS_PATH, TRAIN_CONFIG)
    print('設定保存完了:', SETTINGS_PATH)
else:
    print('表示中の設定はDriveから復元/このセッションのみ。変更時はSAVE_SETTINGS=Trueで保存してください。')
print(json.dumps(TRAIN_CONFIG,ensure_ascii=False,indent=2))


## 8. CPU: モデルキャッシュ確認・取得ゲート

固定revisionをDrive上のHub cacheへ保存し、再実行時は再利用します。ALLOW_MODEL_DOWNLOAD=Falseのときはローカルキャッシュだけを探し、ネットワークアクセスしません。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

from huggingface_hub import snapshot_download
from modeling import RELATIVE_MODEL_ID,METRIC_OUTDOOR_MODEL_ID,RELATIVE_MODEL_REVISION,METRIC_OUTDOOR_MODEL_REVISION
ALLOW_MODEL_DOWNLOAD = False  # 重み取得を許可する場合のみTrue
MODEL_PATHS = {}
specs = {
 'relative':(RELATIVE_MODEL_ID,RELATIVE_MODEL_REVISION),
 'metric':(METRIC_OUTDOOR_MODEL_ID,METRIC_OUTDOOR_MODEL_REVISION)
}
needed = sorted({'relative' if m=='relative_encoder' else 'metric' for m in TRAINING_MODES})
for key in needed:
    repo,revision = specs[key]
    try:
        path = snapshot_download(repo_id=repo,revision=revision,cache_dir=str(MODEL_CACHE_DIR),
            local_files_only=not ALLOW_MODEL_DOWNLOAD,allow_patterns=['config.json','*.safetensors'])
        MODEL_PATHS[key] = str(path)
        print('モデルcache確認成功:',key,path)
    except Exception as exc:
        print('モデル未取得/取得失敗:',repo,revision,'|',type(exc).__name__,str(exc))
        print('取得許可:',ALLOW_MODEL_DOWNLOAD,'| cacheに無ければ取得を許可するときだけALLOW_MODEL_DOWNLOAD=Trueにして、このセルのみ再実行してください。通信失敗時はモデルrevision、Drive空き容量、ネットワークを確認してください。')

## 9. GPU: CUDA・メモリ事前計測

ここからはGPUを使用します。CUDAが使えるColab runtimeを選び、このセルを個別に実行してください。合成1 stepのみで、UAVデータは読み込みません。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_GPU_PREFLIGHT = False
PREFLIGHT_REPORTS = {}
if RUN_GPU_PREFLIGHT:
    import torch
    refresh_preparation_state()
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable。Colab runtimeをGPUに変更してください。CPU準備は完了しています。')
    if MAX_DEPTH_M is None or not AUDIT_OK or GT_REPORT is None: raise RuntimeError('GT max_depth設定とデータ監査passが必要です。')
    if not all(('relative' if m=='relative_encoder' else 'metric') in MODEL_PATHS for m in TRAINING_MODES):
        raise RuntimeError('選択方式のモデル重みがありません。前のモデルキャッシュセルを確認してください。')
    print('GPU:',torch.cuda.get_device_name(0),'| total GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2),
          '| free GiB:',round(torch.cuda.mem_get_info()[0]/1024**3,2),
          '| /content free GiB:',round(shutil.disk_usage('/content').free/1024**3,2))
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    relative_path=MODEL_PATHS.get('relative',RELATIVE_MODEL_ID)
    metric_path=MODEL_PATHS.get('metric',METRIC_OUTDOOR_MODEL_ID)
    for mode in TRAINING_MODES:
        report=PREFLIGHT_DIR/f'{mode}_res{IMAGE_SIZE}_bs{BATCH_SIZE}_{stamp}.json'
        cmd=[sys.executable,str(CODE_DIR/'memory_preflight.py'),'--initialization',mode,
             '--relative-model',relative_path,'--metric-model',metric_path,'--image-size',str(IMAGE_SIZE),
             '--batch-size',str(BATCH_SIZE),'--max-depth',str(MAX_DEPTH_M),'--output',str(report),
             '--amp' if USE_AMP else '--no-amp']
        subprocess.run(cmd,check=True)
        PREFLIGHT_REPORTS[mode]=report
else: print('RUN_GPU_PREFLIGHT=False: GPUを使いません。')

## 10. GPU: 試験学習・本学習

まずEPOCHS=1の短い試験runを推奨します。学習前にmanifest、ファイル存在、GT設定、モデル、同一設定の事前計測が必須です。1epochごとにDriveへチェックポイントを保存し、時間見積もりと残りepochを表示します。testは読み込みません。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_TRAINING = False
if RUN_TRAINING:
    import torch, hashlib
    refresh_preparation_state()
    if not torch.cuda.is_available(): raise RuntimeError('このセルはCUDAが必要です。')
    if not AUDIT_OK or GT_REPORT is None or MAX_DEPTH_M is None: raise RuntimeError('audit passとtrain-only GT/max_depthが必要です。')
    train_manifest_sha=hashlib.sha256((PLAN_DIR/'train.csv').read_bytes()).hexdigest()
    if GT_REPORT.get('source_manifest_sha256')!=train_manifest_sha: raise RuntimeError('GT解析とtrain manifestが一致しません。GTを再解析してください。')
    if AUDIT_REPORT.get('manifest_sha256',{}).get('train')!=train_manifest_sha or AUDIT_REPORT.get('manifest_sha256',{}).get('validation')!=hashlib.sha256((PLAN_DIR/'validation.csv').read_bytes()).hexdigest(): raise RuntimeError('監査対象manifestと学習manifestが一致しません。監査を再実行してください。')
    if set(PREFLIGHT_REPORTS)!=set(TRAINING_MODES): raise RuntimeError('各選択方式のGPU事前計測を先にpassさせてください。')
    if not RUN_LABEL or '/' in RUN_LABEL or chr(92) in RUN_LABEL: raise ValueError('RUN_LABELを単一フォルダー名にしてください。')
    preparation_context(manifests={'train':PLAN_DIR/'train.csv','validation':PLAN_DIR/'validation.csv'}, rgb_root=ACTIVE_RGB_ROOT, depth_root=ACTIVE_DEPTH_ROOT, physical_splits=['train','validation'])
    train_count=plan_summary['selected_pair_counts']['train']
    step_count=(train_count+BATCH_SIZE-1)//BATCH_SIZE
    print(f'{train_count} pairs/epoch; optimizer steps/epoch={(step_count+GRAD_ACCUMULATION-1)//GRAD_ACCUMULATION}; epochs={EPOCHS}')
    print('初回epoch終了後、実測時間から総所要時間と残り時間を概算します。')
    relative_path=MODEL_PATHS.get('relative',RELATIVE_MODEL_ID)
    metric_path=MODEL_PATHS.get('metric',METRIC_OUTDOOR_MODEL_ID)
    for mode in TRAINING_MODES:
        run_dir=RUNS_DIR/f'{mode}_{RUN_LABEL}'
        cmd=[sys.executable,str(CODE_DIR/'train.py'),'--train-manifest',str(PLAN_DIR/'train.csv'),
             '--val-manifest',str(PLAN_DIR/'validation.csv'),'--rgb-root',str(ACTIVE_RGB_ROOT),
             '--depth-root',str(ACTIVE_DEPTH_ROOT),'--initialization',mode,'--relative-model',relative_path,
             '--metric-model',metric_path,'--max-depth',str(MAX_DEPTH_M),'--image-size',str(IMAGE_SIZE),
             '--batch-size',str(BATCH_SIZE),'--grad-accumulation',str(GRAD_ACCUMULATION),
             '--epochs',str(EPOCHS),'--run-kind',RUN_KIND,'--keep-recent-checkpoints','0','--workers',str(WORKERS),'--seed',str(SEED),
             '--save-every-epochs','1','--memory-preflight-report',str(PREFLIGHT_REPORTS[mode]),
             '--output-dir',str(run_dir),'--amp' if USE_AMP else '--no-amp']
        if RESUME_TRAINING: cmd.append('--resume')
        subprocess.run(cmd,check=True)
else: print('RUN_TRAINING=False: 学習を開始していません。')

## 10b. CPUでも確認可能: 試験結果・checkpoint・時間概算

このセルは学習を開始しません。試験後に損失、validation各指標、時間、実測VRAMを確認してください。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('最初のDrive接続・状態復元セルを実行してください。')
ESTIMATE_EPOCHS = 20  # 時間の概算だけ。学習は開始しません。
for mode in TRAINING_MODES:
    run_dir = RUNS_DIR/f'{mode}_{RUN_LABEL}'
    log = run_dir/'epoch_metrics.jsonl'
    if not log.is_file():
        print('学習結果はまだありません:',run_dir)
        continue
    rows = [json.loads(line) for line in log.read_text(encoding='utf-8').splitlines() if line.strip()]
    row = rows[-1]
    print(mode, '| epoch:',row['epoch'],'| SiLog:',row['train_silog'])
    print('validation:',row['validation'])
    print('epoch秒:',row['epoch_seconds'],'| peak allocated/reserved GiB:',
          row.get('peak_allocated_bytes',0)/1024**3, row.get('peak_reserved_bytes',0)/1024**3)
    mean = sum(r['epoch_seconds'] for r in rows)/len(rows)
    print(f'{ESTIMATE_EPOCHS}epochの概算: {mean*ESTIMATE_EPOCHS/60:.1f}分（checkpoint保存・接続待ち等は別。CCU消費量ではありません）')
    print('run状態:',(run_dir/'run_status.json').read_text(encoding='utf-8'))
    print('checkpoint:',run_dir/'latest_checkpoint.json',run_dir/'best_checkpoint.json')


## 11. 最終段階: 独立test（1回）

モデル・設定を確定し、independent_testを別途取得・監査した後だけ実行します。testを見てモデルを選び直した場合、そのtestは独立評価ではなくなります。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_INDEPENDENT_TEST = False
TEST_MODE = 'relative_encoder'
if RUN_INDEPENDENT_TEST:
    if MAX_DEPTH_M is None: raise RuntimeError('先にtrain-only GT解析と学習設定を確認してください。')
    if not (RUNS_DIR/f'{TEST_MODE}_{RUN_LABEL}'/'best_checkpoint.json').is_file(): raise RuntimeError('最終モデルがありません。試験結果・validationによるモデル選択を完了してから独立testを実行してください。')
    import hashlib
    if TEST_MODE not in TRAINING_MODES: raise ValueError('TEST_MODEは学習済み方式にしてください。')
    run_dir=RUNS_DIR/f'{TEST_MODE}_{RUN_LABEL}'
    best=json.loads((run_dir/'best_checkpoint.json').read_text(encoding='utf-8'))
    checkpoint=run_dir/best['path']
    test_audit=WORK_DIR/f'test_audit_{datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")}.json'
    subprocess.run([sys.executable,str(CODE_DIR/'audit_data.py'),'--train',str(PLAN_DIR/'train.csv'),
        '--validation',str(PLAN_DIR/'validation.csv'),'--independent-test',str(PLAN_DIR/'independent_test.csv'),
        '--physical-splits','train','validation','independent_test','--plan-summary',str(PLAN_DIR/'plan_summary.json'),
        '--rgb-root',str(DRIVE_RGB_ROOT),'--depth-root',str(DRIVE_DEPTH_ROOT),'--output',str(test_audit)],check=True)
    if json.loads(test_audit.read_text(encoding='utf-8')).get('status')!='pass': raise RuntimeError('testを含む物理監査がpassしていません。')
    output=WORK_DIR/f'{TEST_MODE}_{RUN_LABEL}_independent_test.json'
    subprocess.run([sys.executable,str(CODE_DIR/'evaluate.py'),'--model',str(checkpoint),
        '--manifest',str(PLAN_DIR/'independent_test.csv'),'--rgb-root',str(DRIVE_RGB_ROOT),
        '--depth-root',str(DRIVE_DEPTH_ROOT),'--image-size',str(IMAGE_SIZE),
        '--max-depth',str(MAX_DEPTH_M),'--output',str(output)],check=True)
else: print('RUN_INDEPENDENT_TEST=False: testは実行していません。')